<a href="https://colab.research.google.com/github/ma-hirye/Published/blob/main/Col10_VegUrb_Step1_VectorHoles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initials

In [ ]:
## Authenticate GEE
import ee
# Authenticate to your Google Earth Engine account
ee.Authenticate()
ee.Initialize(project='ee-mburb')

## Mount drive
from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import geemap as geemap
import geopandas as gpd
import pandas as pd
import time

# !pip install fiona

import os
import datetime
import pytz
from glob import glob
import geemap

In [ ]:
# ===============================
# User settings
# ===============================


## MapBiomas Dataset and scale
Col_all = ee.Image('projects/mapbiomas-public/assets/brazil/lulc/collection10/mapbiomas_brazil_collection10_coverage_v2')
print("Col_all: ", Col_all.getInfo())
escala = 30


## Get Grids
Cartas = ee.FeatureCollection("projects/ee-mburb/assets/BASES/Cartas_1mi_4326")
print(f'Cartas size: {Cartas.size().getInfo()}')
Cartas_list = Cartas.aggregate_array("grid_name").getInfo()
print("Cartas_list: ", Cartas_list)
# Cartas_list = [
#     'NA-19','NA-20',
#     'NA-21','NA-22',
#     'NB-20','NB-21','NB-22',
#     'SA-19','SA-20','SA-21','SA-22','SA-23','SA-24',
#     'SB-18','SB-19','SB-20','SB-21','SB-22','SB-23','SB-24','SB-25',
#     'SC-18','SC-19','SC-20','SC-21','SC-22','SC-23','SC-24','SC-25',
#     'SD-20','SD-21','SD-22','SD-23','SD-24',
#     'SE-20','SE-21',
#     'SE-22','SE-23','SE-24',
#     'SF-21','SF-22','SF-23','SF-24',
#     'SG-21','SG-22','SG-23',
#     'SH-21','SH-22',
#     'SI-22'
#     ]
# Cartas_list = Cartas_list[0:2]
# Cartas_list = ['SD-20','SD-21']
# Cartas_list = ['SF-23']
# Cartas_list = ['SC-20']
print("Cartas_list: ", len(Cartas_list))


## Define and create output folders
output_aus_folder = '/content/drive/MyDrive/COLAB_Col10_test2/AUs/'
os.makedirs(output_aus_folder, exist_ok=True)
output_bbs_folder = '/content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/'
os.makedirs(output_bbs_folder, exist_ok=True)
output_aus_holes_folder = '/content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/'
os.makedirs(output_aus_holes_folder, exist_ok=True)


## Define years list
start_year = 1985
end_year = 2024
years_list = [str(year) for year in range(start_year, end_year + 1)]
years_list = ['2024']
print("years_list: ", years_list)

Col_all:  {'type': 'Image', 'bands': [{'id': 'classification_1985', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [154468, 146235], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -74.02073025380652, 0, -0.00026949458523585647, 5.405791885246045]}, {'id': 'classification_1986', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [154468, 146235], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -74.02073025380652, 0, -0.00026949458523585647, 5.405791885246045]}, {'id': 'classification_1987', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [154468, 146235], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -74.02073025380652, 0, -0.00026949458523585647, 5.405791885246045]}, {'id': 'classification_1988', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [154468, 146235], 

# Run and export AUs

## Functions

In [ ]:
# =======================================================
# Functions to get AUs
# =======================================================

def export_AUs(LIST_GRIDS, GRIDS, IM, SCALE, OUTPUT_PATH_AU):
    for grid in LIST_GRIDS:
        print(f'---------------- {grid}')

        if os.path.exists(f"{OUTPUT_PATH_AU}{grid}.shp"):
          print(f"{OUTPUT_PATH_AU}{grid}.shp already exists, skipping.")
        else:
          print(f"{OUTPUT_PATH_AU}{grid}.shp not found, processing.")

          # print(GRIDS.aggregate_array('grid_name').getInfo())
          Carta = GRIDS.filter(ee.Filter.eq("grid_name", grid)).first()
          # print(f'Carta: {Carta}')

          im_urban = IM.eq(24).selfMask()
          # print(im_urban.arrayLengths().getInfo())

          bbs_polygons = im_urban.reduceToVectors(
              geometry=Carta.geometry(),
              scale=ee.Number(SCALE),
              maxPixels=10000000000,
              eightConnected=False,
              tileScale=1,
              geometryType="polygon"
          )
          print("bbs_polygons size:", bbs_polygons.size().getInfo())
          geemap.ee_to_shp(bbs_polygons, filename=OUTPUT_PATH_AU + grid + '.shp')
          print(f'Saved {OUTPUT_PATH_AU}{grid}.shp')

print("export_AUs(LIST_GRIDS, GRIDS, IM, SCALE, OUTPUT_PATH_AU)")

export_AUs(LIST_GRIDS, GRIDS, IM, SCALE, OUTPUT_PATH_AU)


## Run

In [ ]:
# STEP 1: Run and export AUs per grid/year to output_aus_folder
# STEP 2: Merge and export AUs per year to shapefile in

In [ ]:
# ==============================================================
# STEP 1: Run and export AUs per grid/year
# ==============================================================
print("####################### STEP 2: Run and export AUs / AUsHoles per year #######################")

for year in years_list:
    print("####################### year:", year)

    im = Col_all.select('classification_' + year)

    output_aus_path = os.path.join(output_aus_folder, f"Y{year}/")
    os.makedirs(output_aus_path, exist_ok=True)

    export_AUs(Cartas_list, Cartas, im, escala, output_aus_path)

In [ ]:
# ==============================================================
# STEP 2: Merge and export AUs per year
# ==============================================================
print("####################### STEP 3: Merge and export AUs per year #######################")

for year in years_list:
  print("####################### year:", year)

  output_ausall_file = f"{output_aus_folder}/AllGrids_{year}.shp"

  if not os.path.exists(output_ausall_file):
      print(f"AllGrids_{year} does not exist, , processing...")

      im = Col_all.select('classification_'+year);
      # print(im.getInfo())

      # Find all .shp files in the folder
      shapefiles = glob(os.path.join(output_aus_path, "*.shp"))

      # Read and store them in a list of GeoDataFrames
      merged = gpd.GeoDataFrame(
          pd.concat((gpd.read_file(shp) for shp in shapefiles), ignore_index=True),
          crs=gpd.read_file(shapefiles[0]).crs
      )
      print(f"  -> Merge completed: {len(merged)} features")

      # dissolve and explode all grids of buffers
      dissolved_gdf = merged.dissolve()
      print(f"Dissolve completed")

      exploded_gdf = dissolved_gdf.explode(index_parts=True)
      print(f"Explode completed")

      # Add IDs
      gdf_ids = exploded_gdf.reset_index(drop=True)
      gdf_ids["id"] = range(1, len(gdf_ids) + 1)
      # display(gdf_ids)
      print(f"IDs added")

      # Save merged shapefile (optional)
      gdf_ids.to_file(output_ausall_file)
      print(f"Merged shapefile saved at: {output_ausall_file}")

  else:
      print(f"AllGrids_{year} already exists, skipping.")

####################### STEP 3: Merge and export AUs per year #######################
####################### year: 2024
AllGrids_2024 does not exist, , processing...
  -> Merge completed: 1585 features
Dissolve completed
Explode completed
IDs added
Merged shapefile saved at: /content/drive/MyDrive/COLAB_Col10_test2/AUs//AllGrids_2024.shp


# Run and export AUs+Holes

## Functions

In [ ]:
# =======================================================
# Functions to Step 1: Run and export BBs per grid/year
# =======================================================

def create_buffer_feature(feature):
    # feature is an ee.Feature
    geom_buffered = feature.geometry().buffer(1000)
    return ee.Feature(geom_buffered)
print("create_buffer_feature(feature)")


def export_BBs(LIST_GRIDS, GRIDS, IM, SCALE, OUTPUT_PATH_BB, YEAR):
    bbs_buf_all = ee.FeatureCollection([])

    for grid in LIST_GRIDS:
        print(f'---------------- {grid}')

        # print(GRIDS.aggregate_array('grid_name').getInfo())
        Carta = GRIDS.filter(ee.Filter.eq("grid_name", grid)).first()
        # print(f'Carta: {Carta}')

        im_urban = IM.eq(24).selfMask()
        # print(im_urban.arrayLengths().getInfo())

        bbs_polygons = im_urban.reduceToVectors(
            geometry=Carta.geometry(),
            scale=ee.Number(SCALE),
            maxPixels=10000000000,
            eightConnected=False,
            tileScale=1,
            geometryType="polygon"
        )

        bbs = bbs_polygons.map(lambda f: f.bounds())
        bbs_buf = bbs.map(create_buffer_feature)
        bbs_buf = ee.FeatureCollection([ee.Feature(bbs_buf.union())])
        bbs_buf = bbs_buf.flatten()
        print("bbs_buf size:", bbs_buf.size().getInfo())
        geemap.ee_to_shp(bbs_buf, filename=OUTPUT_PATH_BB + grid + '.shp')
        print(f'Saved {OUTPUT_PATH_BB}{grid}.shp')

    return
print("export_BBs_AUs(LIST_GRIDS, GRIDS, IM, SCALE, OUTPUT_PATH_BB, YEAR)")

create_buffer_feature(feature)
export_BBs_AUs(LIST_GRIDS, GRIDS, IM, SCALE, OUTPUT_PATH_BB, YEAR)


In [ ]:
# =======================================================
# Functions to get Holes
# =======================================================

def group_polygons_by_area(gdf, n_largest=5):
    """
    Group polygons by area.

    Parameters
    ----------
    gdf : GeoDataFrame
        Input GeoDataFrame with polygon geometries.
    n_largest : int
        Number of largest polygons that should each form their own group.

    Returns
    -------
    GeoDataFrame
        Original GeoDataFrame with columns:
        - 'area'
        - 'group'
    """

    gdf = gdf.copy()
    original_crs = gdf.crs

    # Project if needed for correct area calculation
    if gdf.crs.is_geographic:
        gdf = gdf.to_crs(gdf.estimate_utm_crs())

    # Calculate area
    gdf["area"] = gdf.geometry.area

    # Sort polygons
    gdf = gdf.sort_values("area", ascending=False).reset_index(drop=True)

    # Target area (area of the nth largest polygon)
    target_area = gdf.loc[n_largest - 1, "area"]

    group_ids = []
    current_group = n_largest
    current_sum = 0

    for i, area in enumerate(gdf["area"]):

        if i < n_largest:
            group_ids.append(i)
        else:
            if current_sum + area > target_area:
                current_group += 1
                current_sum = 0

            group_ids.append(current_group)
            current_sum += area

    gdf["group"] = group_ids

    # Return to original CRS
    gdf = gdf.to_crs(original_crs)

    return gdf


def export_ShpHoles_byGrid_Colab4_6933(IM, GROUP_LIST, GROUP_NAME, BBs, SCALE, OUTPUT_PATH, year):
## functions to export shpHoles FINAL- without area, without buffer, by id, form COLAB
## EPSG:6933 (EASE-Grid 2.0 Global) - It maintains the true area of geographic features, which is essential for scientific applications and analyses where accurate representation of size is needed.

    im_urban = IM.updateMask(IM.eq(24))
    im_nurban_all = IM.updateMask(IM.neq(24)).selfMask()
    im_nurban = im_urban.unmask().remap([0,1],[1,0]).selfMask()
    im_AU = im_urban.unmask()

    im_Holes_list = ee.List([])

    output_file = f'{OUTPUT_PATH}Holes_{str(year)}_{GROUP_NAME}.shp'

    if os.path.exists(output_file):
      print(f"Holes_{str(year)}_{GROUP_NAME}.shp already exists, skipping.")
    else:
      print(f"Holes_{str(year)}_{GROUP_NAME}.shp not found, processing.")

      vector_all = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

      for ID in GROUP_LIST:
          # print(f"Processing ID: {ID}")
          feat_sel = BBs[BBs['id'] == ID]

          feat_sel_proj = feat_sel.to_crs("EPSG:6933")
          inner_3857 = feat_sel_proj.geometry.buffer(-1)
          inner_3857_gdf = gpd.GeoDataFrame(geometry=inner_3857, crs="EPSG:6933")
          border_3857_gdf = feat_sel_proj.difference(inner_3857_gdf)
          border_gdf = border_3857_gdf.to_crs(feat_sel.crs)
          # print(f"border_gdf")

          feat = geemap.gdf_to_ee(feat_sel)
          # print("feat: ", feat.getInfo())

          vector_tmp = ee.Image(im_nurban).reduceToVectors(
              geometry= feat.geometry(),
              scale=escala,
              maxPixels=10000000000,
              eightConnected=True,
              geometryType='polygon',
              bestEffort=False,
              tileScale = 1
              )
          # print('vector done')

          vector_holes_size = vector_tmp.size().getInfo()
          # print("vector_holes_size:", vector_holes_size)

          if vector_holes_size > 0:

            gdf_tmp_all = geemap.ee_to_gdf(vector_tmp)

            border_union = border_gdf.union_all()
            gdf_tmp = gdf_tmp_all[~gdf_tmp_all.geometry.intersects(border_union)] # Keep only geometries that intersect the border

            gdf_tmp_3857 = gdf_tmp.to_crs("EPSG:6933") #Project to meters (e.g., EPSG:3857) for accurate area calculation
            gdf_tmp_3857['area_m2'] = gdf_tmp_3857.geometry.area.astype(int)
            gdf_tmp_4326 = gdf_tmp_3857.to_crs("EPSG:4326")

            gdf_tmp_4326['id'] = int(ID)
            gdf_tmp_4326['grid'] = GROUP_NAME

            vector_all = pd.concat([vector_all, gdf_tmp_4326], ignore_index=True)

      print("no. of Holes: ", len(vector_all))


      if len(vector_all) > 0:
          vector_all.to_file(output_file)
          print(f'✅ Shapefile saved: {output_file}.')
      else:
          print(f"No vector found.")

## Run

In [ ]:
# STEP 1: Run and export BBs per grid/year to output_bbs_folder/year
# STEP 2: Merge and export BBs per year shapefile to output_bbs_folder/year/AllGrids
# STEP 3: Run and export AUsHoles per group/year to output_holes_folder/year
# STEP 4: Merge and export AUsHoles per year

In [ ]:
# ==============================================================
# STEP 1: Run and export BBs per grid/year
# ==============================================================
print("####################### STEP 1: Run and export BBs per year #######################")

for year in years_list:
    print("####################### year:", year)

    im = Col_all.select('classification_' + year)

    output_bbs_path = os.path.join(output_bbs_folder, f"Y{year}/")
    os.makedirs(output_bbs_path, exist_ok=True)
    print(f'output_bbs_path: {output_bbs_path}')

    export_BBs(Cartas_list, Cartas, im, escala, output_bbs_path, year)

####################### STEP 1: Run and export BBs per year #######################
####################### year: 2024
output_bbs_path: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/
---------------- SA-21
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SA-21.shp
---------------- NA-19
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/NA-19.shp
---------------- SC-20
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SC-20.shp
---------------- SB-21
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SB-21.shp
---------------- NA-21
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/NA-21.shp
---------------- SD-22
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SD-22.shp
---------------- SB-18
bbs_buf size: 1
Save

Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SD-24.shp
---------------- SA-20
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SA-20.shp
---------------- SD-20
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SD-20.shp
---------------- SF-23
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SF-23.shp
---------------- SF-24
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SF-24.shp
---------------- SF-21
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SF-21.shp
---------------- SF-22
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SF-22.shp
---------------- SC-24
bbs_buf size: 1
Saved /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/SC-24.shp
---------------- SC-25
bbs_buf size: 1


In [ ]:
# ==============================================================
# STEP 2: Merge and export BBs per year
# ==============================================================
print("####################### STEP 2: Merge and export AUs per year #######################")

for year in years_list:
  print("####################### year:", year)

  output_bbs_path = os.path.join(output_bbs_folder, f"Y{year}/")
  output_ausall_file = f"{output_bbs_path}AllGrids_{year}.shp"

  if not os.path.exists(output_ausall_file):
      print(f"AllGrids_{year} does not exist, processing...")

      im = Col_all.select('classification_'+year);
      # print(im.getInfo())

      # Find all .shp files in the folder
      shapefiles = glob(os.path.join(output_bbs_path, "*.shp"))

      # Read and store them in a list of GeoDataFrames
      merged = gpd.GeoDataFrame(
          pd.concat((gpd.read_file(shp) for shp in shapefiles), ignore_index=True),
          crs=gpd.read_file(shapefiles[0]).crs
      )
      print(f"  -> Merge completed: {len(merged)} features")

      # dissolve and explode all grids of buffers
      dissolved_gdf = merged.dissolve()
      print(f"Dissolve completed")

      exploded_gdf = dissolved_gdf.explode(index_parts=True)
      print(f"Explode completed")

      # Add IDs
      gdf_ids = exploded_gdf.reset_index(drop=True)
      gdf_ids["id"] = range(1, len(gdf_ids) + 1)
      # display(gdf_ids)
      print(f"IDs added")

      # Save merged shapefile (optional)
      gdf_ids.to_file(output_ausall_file)
      print(f"Merged shapefile saved at: {output_ausall_file}")

  else:
      print(f"AllGrids_{year} already exists, skipping.")

####################### STEP 2: Merge and export AUs per year #######################
####################### year: 2024
AllGrids_2024 does not exist, processing...
  -> Merge completed: 49 features
Dissolve completed
Explode completed
IDs added
Merged shapefile saved at: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/AllGrids_2024.shp


In [ ]:
# ==============================================================
# STEP 3: Run and export AUsHoles per group/year
# ==============================================================
print("####################### STEP 3: Run and export AUsHoles per year #######################")

for year in years_list:
    print("####################### year:", year)
    # Construct the year-specific output folder path using the base folder
    current_output_holes_folder = os.path.join(output_aus_holes_folder, f"Y{year}/")
    os.makedirs(current_output_holes_folder, exist_ok=True) # Ensure the directory exists

    im = Col_all.select('classification_' + year)

    output_bbsall_file = os.path.join(output_bbs_folder, f"Y{year}/AllGrids_{year}.shp")
    print(" output_bbsall_file: ", output_bbsall_file)

    gdf_ids = gpd.read_file(output_bbsall_file)
    print("size of gdf_ids: ", len(gdf_ids))

    gdf_grouped = group_polygons_by_area(gdf_ids, n_largest=5)
    # Iterate directly over the groups returned by groupby to get group_id and group_df
    print("Number of groups: ", gdf_grouped["group"].nunique())

    for group_id, group_df in gdf_grouped.groupby("group"):
        group_name = f"Group_{group_id}"

        print(f"Processing group {group_id} - size: {len(gdf_grouped_group)}")

        gdf_ids_list = sorted(group_df['id'].dropna().unique().tolist())
        print("size of gdf_ids_list for this group: ", len(gdf_ids_list))

        export_ShpHoles_byGrid_Colab4_6933(im, gdf_ids_list, group_name, group_df, escala, current_output_holes_folder, year)


####################### STEP 3: Run and export AUsHoles per year #######################
####################### year: 2024
 output_bbsall_file:  /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step1and2_BBs/Y2024/AllGrids_2024.shp
size of gdf_ids:  23450
Number of groups:  105
Processing group 0 - size: 1
size of gdf_ids_list for this group:  1
Holes_2024_Group_0.shp not found, processing.
no. of Holes:  2092
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_0.shp.
Processing group 1 - size: 1
size of gdf_ids_list for this group:  1
Holes_2024_Group_1.shp not found, processing.
no. of Holes:  669
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_1.shp.
Processing group 2 - size: 1
size of gdf_ids_list for this group:  1
Holes_2024_Group_2.shp not found, processing.
no. of Holes:  1384
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_

no. of Holes:  211
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_54.shp.
Processing group 55 - size: 122
size of gdf_ids_list for this group:  122
Holes_2024_Group_55.shp not found, processing.
no. of Holes:  299
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_55.shp.
Processing group 56 - size: 128
size of gdf_ids_list for this group:  128
Holes_2024_Group_56.shp not found, processing.
no. of Holes:  302
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_56.shp.
Processing group 57 - size: 135
size of gdf_ids_list for this group:  135
Holes_2024_Group_57.shp not found, processing.
no. of Holes:  262
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_57.shp.
Processing group 58 - size: 142
size of gdf_ids_list for this group:  142
Holes_2024_

no. of Holes:  89
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_83.shp.
Processing group 84 - size: 462
size of gdf_ids_list for this group:  462
Holes_2024_Group_84.shp not found, processing.
no. of Holes:  61
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_84.shp.
Processing group 85 - size: 485
size of gdf_ids_list for this group:  485
Holes_2024_Group_85.shp not found, processing.
no. of Holes:  74
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_85.shp.
Processing group 86 - size: 510
size of gdf_ids_list for this group:  510
Holes_2024_Group_86.shp not found, processing.
no. of Holes:  66
✅ Shapefile saved: /content/drive/MyDrive/COLAB_Col10_test2/AUsHoles_Step3and4_AUsHoles/Y2024/Holes_2024_Group_86.shp.
Processing group 87 - size: 534
size of gdf_ids_list for this group:  534
Holes_2024_Grou

In [ ]:
# ==============================================================
# STEP 4: Merge and export AUsHoles per year
# ==============================================================
print("####################### STEP 4: Merge and export AUsHoles per year #######################")

for year in years_list:
  print("####################### year:", year)

  output_ausall_file = f"{output_holes_folder}/AllGrids_{year}.shp"

  if not os.path.exists(output_ausall_file):
      print(f"AllGrids_{year} does not exist, , processing...")

      # Find all .shp files in the folder
      shapefiles = glob(os.path.join(output_holes_folder, "*.shp"))

      # Read and store them in a list of GeoDataFrames
      merged = gpd.GeoDataFrame(
          pd.concat((gpd.read_file(shp) for shp in shapefiles), ignore_index=True),
          crs=gpd.read_file(shapefiles[0]).crs
      )
      print(f"  -> Merge completed: {len(merged)} features")

      # Save merged shapefile (optional)
      merged.to_file(output_ausall_file)
      print(f"Merged shapefile saved at: {output_ausall_file}")

  else:
      print(f"AllGrids_{year} already exists, skipping.")
